# ViT-Ctr Phase 3: Full Training on AutoDL
运行前请确认已选择GPU实例（RTX 3090/4090推荐）

In [ ]:
# 环境检查
import torch
import sys
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# CRITICAL: 数据集可用性检查 — Wave 0 Gate
import os
import h5py

# AutoDL数据路径
DATA_DIR = '/root/autodl-tmp/data'
REQUIRED_FILES = ['dithioester.h5', 'trithiocarbonate.h5', 'xanthate.h5', 'dithiocarbamate.h5']
MIN_SAMPLES_PER_FILE = 100_000  # 完整数据集每文件约250K

all_ok = True
for fname in REQUIRED_FILES:
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        print(f"[FAIL] File missing: {fpath}")
        all_ok = False
    else:
        with h5py.File(fpath, 'r') as f:
            n = f['fingerprints'].shape[0]
            if n < MIN_SAMPLES_PER_FILE:
                print(f"[WARN] {fname}: only {n} samples (need >= {MIN_SAMPLES_PER_FILE})")
                all_ok = False
            else:
                print(f"[ OK] {fname}: {n} samples")

if not all_ok:
    raise RuntimeError(
        "数据集不完整！训练中止。\n"
        "请先上传Phase 2生成的HDF5文件到AutoDL。\n"
        f"目标路径: {DATA_DIR}"
    )
print("\n✓ 数据集验证通过，可以开始训练")

In [ ]:
# 设置代码路径
import sys
CODE_DIR = '/root/autodl-tmp/ViT-Ctr'
sys.path.insert(0, os.path.join(CODE_DIR, 'src'))
print(f"Code directory: {CODE_DIR}")

In [ ]:
# 训练配置
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# 训练超参数（锁定值）
BATCH_SIZE = 64        # D-02
LR = 3e-4              # D-03
MAX_EPOCHS = 200       # D-05 (早停会提前结束)
NUM_WORKERS = 4        # AutoDL CPU更强，可用更多workers
CHECKPOINT_DIR = '/root/autodl-tmp/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

In [ ]:
# 运行训练
!python {CODE_DIR}/src/train.py \
    --h5_dir {DATA_DIR} \
    --epochs {MAX_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --num_workers {NUM_WORKERS} \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --seed 42

In [ ]:
# 损失曲线可视化
import json
import matplotlib.pyplot as plt

log_path = os.path.join(CHECKPOINT_DIR, 'training_log.json')
if os.path.exists(log_path):
    with open(log_path) as f:
        log = json.load(f)
    epochs = [e['epoch'] for e in log]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, [e['train_loss'] for e in log], label='Train')
    axes[0].plot(epochs, [e['val_loss'] for e in log], label='Val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Weighted MSE Loss')
    axes[0].set_title('Total Loss')
    axes[0].legend()
    
    axes[1].plot(epochs, [e['loss_ctr'] for e in log], label='Ctr (×2.0)')
    axes[1].plot(epochs, [e['loss_inh'] for e in log], label='Inhibition')
    axes[1].plot(epochs, [e['loss_ret'] for e in log], label='Retardation')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Per-output MSE')
    axes[1].set_title('Per-output Loss')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'loss_curves.png'), dpi=150)
    plt.show()
else:
    print(f"Log file not found at {log_path}")

## 训练完成后
1. 下载 `checkpoints/best_model.pth` 到本地
2. 记录最佳验证集loss和达到早停时的epoch数
3. 运行 Bootstrap UQ 进行不确定性量化